# 03 · Pandas Analysis
Ten analytical questions answered with Pandas on the cleaned data in PostgreSQL.
Each answer is followed by a printed fact (computed from the data) and a written insight/recommendation.

## Setup – load tables from PostgreSQL

In [1]:
import pandas as pd
import numpy as np
from db_config import get_engine

engine = get_engine()
pd.set_option("display.float_format", "{:,.2f}".format)

# Pull each table into a DataFrame
movies         = pd.read_sql("SELECT * FROM movies", engine, parse_dates=["release_date"])
genres         = pd.read_sql("SELECT * FROM genres", engine)
movie_genres   = pd.read_sql("SELECT * FROM movie_genres ORDER BY ctid", engine)  # ctid = original load order
cast           = pd.read_sql('SELECT * FROM "cast"', engine)
crew           = pd.read_sql("SELECT * FROM crew", engine)
movie_keywords = pd.read_sql("SELECT * FROM movie_keywords", engine)

# numeric columns come back as Decimal/object from NUMERIC – coerce to float
for c in ["runtime", "popularity", "vote_average"]:
    movies[c] = pd.to_numeric(movies[c], errors="coerce")
movies["release_year"] = movies["release_date"].dt.year
movies.shape

(2503, 13)

## Q1. Total unique movies, genres, cast members and crew members

In [2]:
uniques = pd.Series({
    "movies":       movies["movie_id"].nunique(),
    "genres":       genres["genre_id"].nunique(),
    "cast_members": cast["person_id"].nunique(),     # person_id = one real person
    "crew_members": crew["person_id"].nunique(),
    "keywords":     movie_keywords["keyword_id"].nunique(),
}, name="unique_count")
uniques.to_frame()

,unique_count
movies,2503
genres,19
cast_members,11300
crew_members,2848
keywords,10288


In [3]:
print(f"Avg cast credits per movie: {len(cast)/uniques['movies']:.1f} | "
      f"avg crew credits per movie: {len(crew)/uniques['movies']:.1f}")

Avg cast credits per movie: 9.9 | avg crew credits per movie: 2.9


**Insight:** 2,503 movies, 19 genres, 11,300 unique cast members, 2,848 crew members and 10,288 keywords. Each movie averages ~10 cast credits. **Recommendation:** count people by `person_id`, not name, to avoid merging different people with the same name.

## Q2. Movies with missing or zero budget/revenue – and how to treat them

In [4]:
bad_fin = movies[(movies["budget"].fillna(0) == 0) | (movies["revenue"].fillna(0) == 0)]
summary = pd.Series({
    "total movies":              len(movies),
    "budget = 0 / missing":      (movies["budget"].fillna(0) == 0).sum(),
    "revenue = 0 / missing":     (movies["revenue"].fillna(0) == 0).sum(),
    "either missing":            len(bad_fin),
    "both valid (usable)":       len(movies) - len(bad_fin),
})
print(summary.to_string())
bad_fin[["title", "release_year", "budget", "revenue"]].head(15)

total movies             2503
budget = 0 / missing       90
revenue = 0 / missing     102
either missing            133
both valid (usable)      2370


,title,release_year,budget,revenue
326,Zack Snyder's Justice League,2021,70000000,0
342,Bird Box,2018,19800000,0
407,365 Days,2020,0,9458590
458,All the Bright Places,2020,0,0
474,To All the Boys I've Loved Before,2018,0,0
561,Prey,2022,65000000,0
605,The Kissing Booth,2018,0,0
725,Enola Holmes,2020,21000000,0
731,Extraction,2020,65000000,0
761,"Love, Rosie",2014,0,4439431


In [5]:
# DECISION: 0 means UNKNOWN in TMDB -> convert to NaN (never impute – a guessed budget would fake ROI)
movies["budget_clean"]  = movies["budget"].replace(0, np.nan)
movies["revenue_clean"] = movies["revenue"].replace(0, np.nan)

# Financial subset used for every money-based question below
fin = movies.dropna(subset=["budget_clean", "revenue_clean"]).copy()
print(f"Financial analysis sample: {len(fin):,} of {len(movies):,} movies ({len(fin)/len(movies):.0%})")

Financial analysis sample: 2,370 of 2,503 movies (95%)


**Decision & rationale**
- `0` budget/revenue is treated as **missing (NaN)**, not free/flop.
- These movies are **kept** for non-financial analysis (ratings, genres, talent, keywords) but **excluded** from profit/ROI/averages.
- No imputation: filling with median would invent financials and distort ROI rankings.

**Recommendation:** always report the sample size next to any financial KPI.

## Q3. Profit and ROI for every movie with valid budget & revenue

In [6]:
fin["profit"] = fin["revenue_clean"] - fin["budget_clean"]
fin["roi"]    = fin["profit"] / fin["budget_clean"]          # 1.0 = 100% return
fin[["title", "release_year", "budget_clean", "revenue_clean", "profit", "roi"]] \
    .sort_values("profit", ascending=False).head(15)

,title,release_year,budget_clean,revenue_clean,profit,roi
4,Avatar,2009,"237,000,000.00","2,923,706,026.00","2,686,706,026.00",11.34
16,Avengers: Endgame,2019,"356,000,000.00","2,799,439,100.00","2,443,439,100.00",6.86
19,Titanic,1997,"200,000,000.00","2,264,162,353.00","2,064,162,353.00",10.32
181,Avatar: The Way of Water,2022,"460,000,000.00","2,334,484,620.00","1,874,484,620.00",4.07
65,Star Wars: The Force Awakens,2015,"245,000,000.00","2,068,223,624.00","1,823,223,624.00",7.44
7,Avengers: Infinity War,2018,"300,000,000.00","2,052,415,039.00","1,752,415,039.00",5.84
46,Spider-Man: No Way Home,2021,"200,000,000.00","1,921,206,586.00","1,721,206,586.00",8.61
1780,Zootopia 2,2025,"150,000,000.00","1,868,208,796.00","1,718,208,796.00",11.45
53,Jurassic World,2015,"150,000,000.00","1,671,537,444.00","1,521,537,444.00",10.14
702,Inside Out 2,2024,"200,000,000.00","1,698,863,816.00","1,498,863,816.00",7.49


In [7]:
print(f"Profitable movies: {(fin.profit > 0).mean():.0%} | median ROI: {fin.roi.median():.2f}x | "
      f"total profit: ${fin.profit.sum()/1e9:,.2f}B")

Profitable movies: 92% | median ROI: 2.43x | total profit: $390.79B


**Insight:** 92% of movies with known financials were profitable; total profit is $390.8B. ROI is heavily right-skewed – median 2.43x vs mean 13.2x – because a few hits return 100x+. **Recommendation:** evaluate slates as a portfolio (median ROI + hit rate), not by average ROI alone.

## Q4. Top 15 movies by ROI (budget ≥ $1M)

In [8]:
top_roi = (fin[fin["budget_clean"] >= 1_000_000]
           .nlargest(15, "roi")[["title", "release_year", "budget_clean", "revenue_clean", "roi"]])
top_roi

,title,release_year,budget_clean,revenue_clean,roi
1771,Dragon Ball Super: Broly,2018,"1,000,000.00","125,002,821.00",124.00
542,Snow White and the Seven Dwarfs,1938,"1,488,423.00","184,925,486.00",123.24
1795,The Rocky Horror Picture Show,1975,"1,400,000.00","171,181,400.00",121.27
473,Rocky,1976,"1,000,000.00","117,253,345.00",116.25
1229,Gone with the Wind,1939,"4,000,000.00","402,352,579.00",99.59
724,The Jungle Book,1967,"4,000,000.00","378,000,000.00",93.50
642,Cinderella,1950,"2,900,000.00","263,600,000.00",89.90
374,Saw,2004,"1,200,000.00","104,045,735.00",85.70
727,One Hundred and One Dalmatians,1961,"3,600,000.00","303,000,000.00",83.17
267,E.T. the Extra-Terrestrial,1982,"10,500,000.00","797,307,407.00",74.93


In [9]:
print(f"Median budget of top-15 ROI films: ${top_roi.budget_clean.median()/1e6:,.1f}M "
      f"vs all films ${fin.budget_clean.median()/1e6:,.1f}M")

Median budget of top-15 ROI films: $3.6M vs all films $40.0M


**Insight:** the top-15 ROI films have a median budget of only $3.6M vs $40M for all films – low-budget classics and horror (Rocky, Saw, Insidious, Snow White, Jaws) returning 70–124x. **Recommendation:** allocate part of the slate to low-budget horror/genre films – they deliver outsized percentage returns with limited downside.

## Q5. Movies released per year – peak year

In [10]:
per_year = movies.dropna(subset=["release_year"]).groupby("release_year").size().rename("movies")
peak_year, peak_n = int(per_year.idxmax()), int(per_year.max())
print(f"Peak output year: {peak_year} with {peak_n} movies")
per_year.sort_index(ascending=False).head(15).to_frame()

Peak output year: 2016 with 121 movies


,movies
release_year,
2026,8
2025,26
2024,43
2023,62
2022,65
2021,67
2020,52
2019,100
2018,102


**Insight:** 2016 was the peak year (121 movies). Output falls sharply after 2019 (2020: 52, 2025: 26, 2026: 8), reflecting the pandemic and an incomplete data snapshot for recent years. **Recommendation:** compare year-on-year trends only across fully captured years.

## Q6. Runtime outliers (IQR method)

In [11]:
rt = movies["runtime"].dropna()
q1, q3 = rt.quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
runtime_outliers = movies[(movies["runtime"] < low) | (movies["runtime"] > high)]
print(f"Normal runtime range: {low:.0f}–{high:.0f} min | outliers: {len(runtime_outliers)}")
runtime_outliers[["title", "release_year", "runtime"]].sort_values("runtime", ascending=False)

Normal runtime range: 58–166 min | outliers: 60


,title,release_year,runtime
326,Zack Snyder's Justice League,2021,242.00
1229,Gone with the Wind,1939,238.00
836,Once Upon a Time in America,1984,229.00
1613,Lawrence of Arabia,1962,228.00
1778,Ben-Hur,1959,212.00
594,The Irishman,2019,209.00
1279,Seven Samurai,1954,207.00
1307,Killers of the Flower Moon,2023,206.00
190,The Godfather Part II,1974,202.00
20,The Lord of the Rings: The Return of the King,2003,201.00


**Insight:** half of all films run 99–126 minutes (median 111). 60 films fall outside the normal 58–166 minute range, mostly epics such as Zack Snyder's Justice League (242), Gone with the Wind (238) and Lawrence of Arabia (228). **Recommendation:** long runtimes reduce daily screenings – a real revenue constraint for theatrical releases.

## Q7. Primary genre (first listed) – average popularity

In [12]:
# 'first genre listed' = first row for each movie in the movie_genres bridge (source order)
primary = (movie_genres.groupby("movie_id", sort=False).first().reset_index()
           .merge(genres, on="genre_id"))
movies_pg = movies.merge(primary[["movie_id", "genre_name"]], on="movie_id", how="left") \
                  .rename(columns={"genre_name": "primary_genre"})
pop_by_genre = (movies_pg.groupby("primary_genre")
                .agg(movies=("movie_id", "count"), avg_popularity=("popularity", "mean"),
                     median_popularity=("popularity", "median"))
                .sort_values("avg_popularity", ascending=False))
pop_by_genre

,movies,avg_popularity,median_popularity
primary_genre,,,
Science Fiction,119,21.26,12.65
Music,10,18.32,8.10
Animation,139,16.81,13.67
Family,48,16.71,12.07
Action,474,16.39,11.94
War,23,15.47,11.54
Horror,223,14.85,10.11
Adventure,244,14.79,11.46
Fantasy,93,13.02,10.88


In [13]:
print(f"Most popular primary genre: {pop_by_genre.index[0]} ({pop_by_genre.avg_popularity.iloc[0]:.1f}); "
      f"least: {pop_by_genre.index[-1]} ({pop_by_genre.avg_popularity.iloc[-1]:.1f})")

Most popular primary genre: Science Fiction (21.3); least: Documentary (7.7)


**Insight:** Science Fiction has the highest average popularity (21.3), followed by Music and Animation; Documentary is lowest (7.7). Mean vs median gaps show a few viral titles inflate averages. **Recommendation:** use median popularity when deciding marketing spend by genre.

## Q8. Top 10 most prolific actors + their average movie rating

In [14]:
actor_stats = (cast.merge(movies[["movie_id", "vote_average"]], on="movie_id")
               .groupby(["person_id", "actor_name"])
               .agg(movies=("movie_id", "nunique"), avg_rating=("vote_average", "mean"))
               .reset_index()
               .sort_values(["movies", "avg_rating"], ascending=False)
               .head(10))
actor_stats

,person_id,actor_name,movies,avg_rating
679,2231,Samuel L. Jackson,39,6.95
106,287,Brad Pitt,38,7.26
127,380,Robert De Niro,38,7.24
37,85,Johnny Depp,37,6.86
13,31,Tom Hanks,33,7.29
1272,5293,Willem Dafoe,32,7.11
413,1245,Scarlett Johansson,32,7.06
2674,13240,Mark Wahlberg,32,6.63
161,500,Tom Cruise,31,7.04
64,192,Morgan Freeman,31,6.96


In [15]:
print(f"Top-10 actors avg rating {actor_stats.avg_rating.mean():.2f} vs dataset {movies.vote_average.mean():.2f}")

Top-10 actors avg rating 7.04 vs dataset 6.94


**Insight:** the top-10 actors average 7.04 vs 6.94 for the dataset, but it varies: Tom Hanks (7.29) and Brad Pitt (7.26) are well above, Mark Wahlberg (6.63) below. **Recommendation:** prioritise actors who combine high volume *and* above-average ratings for casting decisions.

## Q9. Titles appearing more than once (duplicates vs remakes)

In [16]:
dup_titles = movies[movies.duplicated("title", keep=False)] \
    .sort_values(["title", "release_year"])[["title", "release_year", "movie_id"]]
dup_summary = (dup_titles.groupby("title")
               .agg(times=("movie_id", "count"),
                    years=("release_year", lambda s: ", ".join(s.dropna().astype(int).astype(str))),
                    distinct_years=("release_year", "nunique")))
dup_summary["type"] = np.where(dup_summary["distinct_years"] > 1, "Remake / different film", "Possible duplicate")
dup_summary.sort_values("times", ascending=False).head(20)

,times,years,distinct_years,type
title,,,,
Robin Hood,3,"1973, 2010, 2018",3,Remake / different film
Halloween,3,"1978, 2007, 2018",3,Remake / different film
A Nightmare on Elm Street,2,"1984, 2010",2,Remake / different film
The Addams Family,2,"1991, 2019",2,Remake / different film
Planet of the Apes,2,"1968, 2001",2,Remake / different film
Point Break,2,"1991, 2015",2,Remake / different film
Poltergeist,2,"1982, 2015",2,Remake / different film
Return,2,"1985, 2011",2,Remake / different film
RoboCop,2,"1987, 2014",2,Remake / different film


In [17]:
print(dup_summary["type"].value_counts().to_string())

type
Remake / different film    49


**Insight:** 49 titles appear more than once, and all are remakes released in different years (e.g. Halloween 1978/2007/2018, Robin Hood 1973/2010/2018) – no true duplicates. **Recommendation:** always join on `movie_id`, never on title.

## Q10. Popularity vs vote count – movies that break the pattern

In [18]:
# Both metrics are skewed -> model the relationship on a log scale, then flag large residuals
d = movies[(movies["vote_count"] > 0) & (movies["popularity"] > 0)].copy()
x, y = np.log1p(d["vote_count"]), np.log1p(d["popularity"])
slope, intercept = np.polyfit(x, y, 1)
d["residual"] = y - (slope * x + intercept)
d["z"] = (d["residual"] - d["residual"].mean()) / d["residual"].std()
anomalies = d[d["z"].abs() > 2.5].copy()
anomalies["pattern"] = np.where(anomalies["z"] > 0, "High popularity, low votes", "Low popularity, high votes")
print(f"corr(log votes, log popularity) = {np.corrcoef(x, y)[0,1]:.2f} | anomalies: {len(anomalies)}")
anomalies[["title", "release_year", "popularity", "vote_count", "pattern"]].sort_values("pattern")

corr(log votes, log popularity) = 0.58 | anomalies: 75


,title,release_year,popularity,vote_count,pattern
35,Spider-Man: Homecoming,2017,169.09,23456,"High popularity, low votes"
46,Spider-Man: No Way Home,2021,438.48,22429,"High popularity, low votes"
62,Spider-Man,2002,180.59,20991,"High popularity, low votes"
89,The Amazing Spider-Man,2012,79.02,18769,"High popularity, low votes"
116,Spider-Man: Far From Home,2019,105.66,17240,"High popularity, low votes"
...,...,...,...,...,...
1771,Dragon Ball Super: Broly,2018,2.07,3216,"Low popularity, high votes"
2096,Return,1985,0.80,2701,"Low popularity, high votes"
2215,Ip Man 2,2010,0.88,2527,"Low popularity, high votes"
2289,Hannibal Rising,2007,1.41,2440,"Low popularity, high votes"


**Insight:** popularity and vote count are moderately related (r = 0.58 on a log scale); 75 movies break the pattern. *High popularity / low votes* = titles currently trending (buzz). *Low popularity / high votes* = older catalogue titles that were once big (e.g. Hannibal Rising, Ip Man 2).

**Recommendation:** target the first group with marketing now (momentum); use the second group for catalogue re-promotion and recommendations.

## Summary of Pandas findings
1. Only a subset of movies has usable financials – every money KPI is reported on that subset.
2. ROI is concentrated in modest-budget films; tentpoles win on absolute profit, not percentage return.
3. Talent volume ≠ quality – combine appearance count with average rating.
4. Popularity anomalies separate *trending new* titles from *legacy catalogue* titles – different marketing plays.